# HintFlow DPO-v2 128 EM（单卡 H100）

**能跑吗？** 能。OSS（`openai/gpt-oss-20b`，mxfp4 ~26GB）+ orch（Qwen3-4B merged ~8GB）同卡起两个 vLLM；调低 `util` / `max_len` / `workers`。预计数小时（比 4×OSS 慢）。

改下面 `WORKDIR` 后 **Run All**。

In [ ]:
import os, subprocess, time, json
from pathlib import Path

WORKDIR = Path(os.environ.get("PAGENT_ROOT", "/content/PAgent")).resolve()
REPO = "https://github.com/Ivann1242/PAgent.git"
OSS_ID = "openai/gpt-oss-20b"
ORCH_ID = "ivaning0919/pagent-hintflow-dpo-v2-merged"
OSS_PORT, ORCH_PORT = 8006, 8086
GPU = "0"
print("WORKDIR=", WORKDIR)

In [ ]:
# 1) clone
if not (WORKDIR / "HintFlow" / "eval_hintflow.py").exists():
    WORKDIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(WORKDIR)])
else:
    print("repo already present")
os.chdir(WORKDIR)
!git -C {WORKDIR} rev-parse --short HEAD

In [ ]:
# 2) env（按机器已有环境可跳过 vllm 重装）
%pip install -q -r requirements.txt huggingface_hub
# 需要 vLLM serve；若本机已装可注释下一行
%pip install -q "vllm>=0.10" 

In [ ]:
# 3) 下载权重
from huggingface_hub import snapshot_download

oss_dir = WORKDIR / "models" / "gpt-oss-20b"
orch_dir = WORKDIR / "checkpoints" / "hintflow_dpo_v2_merged"
oss_dir.parent.mkdir(parents=True, exist_ok=True)
orch_dir.parent.mkdir(parents=True, exist_ok=True)

print("downloading OSS…")
snapshot_download(OSS_ID, local_dir=str(oss_dir))
print("downloading orch…")
snapshot_download(ORCH_ID, local_dir=str(orch_dir))
print("ok", oss_dir, orch_dir)

In [ ]:
# 4) 同卡起 OSS + orch（util 之和 < 1）
import signal, shutil

LOG = WORKDIR / "logs"
LOG.mkdir(exist_ok=True)

def kill_port(port: int):
    subprocess.run(
        f"pkill -f 'vllm.entrypoints.openai.api_server.*--port {port}' || true",
        shell=True,
    )

def wait_model(port: int, name: str, timeout=900):
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=3) as r:
                if name in r.read().decode():
                    print(f":{port} ready ({name})")
                    return
        except Exception:
            pass
        time.sleep(5)
    raise RuntimeError(f"timeout waiting :{port} — see {LOG}")

kill_port(OSS_PORT); kill_port(ORCH_PORT); time.sleep(3)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = GPU

oss_log = open(LOG / "oss-8006.log", "w")
orch_log = open(LOG / "orch-8086.log", "w")

oss_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(oss_dir), "--port", str(OSS_PORT),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.55",
    "--max-model-len", "32768",
    "--max-num-seqs", "4",
    "--served-model-name", "gpt-oss-20b",
]
orch_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(orch_dir), "--port", str(ORCH_PORT),
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.18",
    "--max-model-len", "8192",
    "--max-num-seqs", "4",
    "--served-model-name", "qwen3-4b",
]

print("starting OSS…", flush=True)
oss_p = subprocess.Popen(oss_cmd, env=env, cwd=str(WORKDIR), stdout=oss_log, stderr=subprocess.STDOUT)
wait_model(OSS_PORT, "gpt-oss-20b")

print("starting orch…", flush=True)
orch_p = subprocess.Popen(orch_cmd, env=env, cwd=str(WORKDIR), stdout=orch_log, stderr=subprocess.STDOUT)
wait_model(ORCH_PORT, "qwen3-4b")
print("servers up", oss_p.pid, orch_p.pid)

In [ ]:
# 5) eval 128（单 OSS → workers=4）
data = WORKDIR / "data" / "DAPO-Math.parquet"
assert data.exists(), f"missing {data} — pull latest main"
out = WORKDIR / "checkpoints" / "eval_hintflow_dpo_v2_128"

!python HintFlow/eval_hintflow.py \
  --data-file {data} \
  --limit 128 \
  --workers 4 \
  --solver-urls http://127.0.0.1:{OSS_PORT}/v1 \
  --orch-url http://127.0.0.1:{ORCH_PORT}/v1 \
  --orch-model qwen3-4b \
  --solver-model gpt-oss-20b \
  --out-dir {out}

print((out / "summary.json").read_text())

In [ ]:
# 可选：停服务
# kill_port(OSS_PORT); kill_port(ORCH_PORT)